## Data preprocessing using distributed processing library Ray

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import ray
import re
from transformers import BertTokenizer
import nltk
from nltk.corpus import stopwords


### Defining Preprocessing functions to process data before ML Training

In [ ]:

current_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(current_dir, "..")))

nltk.download("stopwords")
STOPWORDS = stopwords.words("english")

def stratify_split(ds, stratify, test_size):
    return ds.train_test_split(test_size=test_size, shuffle=True)

def tokenize(batch):
    tokenizer = BertTokenizer.from_pretrained("allenai/scibert_scivocab_uncased", return_dict=False)
    encoded_inputs = tokenizer(batch["text"].tolist(), return_tensors="np", padding="longest")
    return dict(ids=encoded_inputs["input_ids"], masks=encoded_inputs["attention_mask"], targets=np.array(batch["tag"]))

def clean_text(text, stopwords=STOPWORDS):
    """Clean raw text string."""
    # Lower
    text = text.lower()

    # Remove stopwords
    pattern = re.compile(r'\b(' + r"|".join(stopwords) + r")\b\s*")
    text = pattern.sub('', text)

    # Spacing and filters
    text = re.sub(r"([!\"'#$%&()*\+,-./:;<=>?@\\\[\]^_`{|}~])", r" \1 ", text)  # add spacing
    text = re.sub("[^A-Za-z0-9]+", " ", text)  # remove non alphanumeric chars
    text = re.sub(" +", " ", text)  # remove multiple spaces
    text = text.strip()  # strip white space at the ends
    text = re.sub(r"http\S+", "", text)  #  remove links

    return text

def preprocess(df, class_to_index):
    """Preprocess the data."""
    df["text"] = df.title + " " + df.description  # feature engineering
    df["text"] = df.text.apply(clean_text)  # clean text
    df = df.drop(columns=["id", "created_on", "title", "description"], errors="ignore")  # clean dataframe
    df = df[["text", "tag"]]  # rearrange columns
    df["tag"] = df["tag"].map(class_to_index)  # label encoding
    outputs = tokenize(df)
    return outputs

### Initialize Ray
In real it distributes tasks with multiple CPUs available in the cluster. Locally it treates each CPU core as single node and launches tasks on every node

In [ ]:
if not ray.is_initialized():
    ray.init()

# Setup execution options to preserve order for deterministic results
ctx = ray.data.DataContext.get_current()
ctx.execution_options.preserve_order = True

### Download Dataset

In [ ]:
from fsspec.implementations.http import HTTPFileSystem

http_fs = HTTPFileSystem()

DATASET_LOC = "https://raw.githubusercontent.com/GokuMohandas/MadeWithML/main/datasets/dataset.csv"

ds = ray.data.read_csv(DATASET_LOC, filesystem=http_fs)

ds.show(limit=5)

### Shuffle and Split Data

In [ ]:
ds = ds.random_shuffle(seed=1234)

print("\n--- Splitting Dataset ---")
test_size = 0.2
train_ds, val_ds = stratify_split(ds, stratify="tag", test_size=test_size)

### Apply Tags to every category (encoding labels)

In [ ]:
print("\n--- Preprocessing Dataset ---")
tags = train_ds.unique(column="tag")
class_to_index = {tag: i for i, tag in enumerate(tags)}
print(f"Class Mapping: {class_to_index}")

### Trigger Preprocessing Task

In [ ]:
# Map the preprocessing function across batches in a distributed manner
sample_ds = train_ds.map_batches(
    preprocess, 
    fn_kwargs={"class_to_index": class_to_index}, 
    batch_format="pandas"
)

# Display the preprocessed result
print("\nPreprocessed Sample:")
sample_ds.show(1)

### Finally Shutdown for gracefull exit

In [ ]:
ray.shutdown()